<a href="https://colab.research.google.com/github/matthew-crouch/kaggle/blob/master/CVPR_Ch3_Cursed_Pixels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔮 Chapter 3 — Cursed Pixels: Adversarial Perturbations

From *Tom Builds, Tom Breaks: Hands-On Attacks and Defenses for Vision-Language Systems* (CVPR half-day tutorial, Pavan Reddy).

After Ch1 + Ch2, Tom's pipeline catches typographic injection (OCR) and anamorphic scaling (LANCZOS + SSIM). FigStep ASR: 4%. Anamorphic ASR: 1%. Security team satisfied.

Then a red-teamer uploads a normal-looking photo. No text. SSIM check passes. OCR finds nothing. But the model's output is harmful, compliant, and detailed.

**The open-source problem.** Tom runs an open-source VLM (LLaVA / SmolVLM / Qwen-VL). The attacker downloads the same weights, computes the gradient of any loss with respect to the input pixels, and updates an invisible perturbation N until the model says exactly what they want it to say. Pure first-order optimization.

Reported ASR (slide 20 / Qi et al. 2024):

| Model            | Clean ASR | Adv ASR | Δ      |
|------------------|-----------|---------|--------|
| LLaVA-1.5-7B     | 3%        | 92%     | +89%   |
| MiniGPT-4        | 5%        | 88%     | +83%   |
| InstructBLIP     | 4%        | 79%     | +75%   |
| → GPT-4V (transfer) | 1%     | 16%     | +15%   |

---

### ⚠️ Ethical Use Notice

This notebook replicates published academic attacks (Goodfellow 2015, Madry 2018, Carlini & Wagner 2017, Moosavi-Dezfooli 2016/2019, Qi et al. 2024, Schlarmann et al. 2024, Zou et al. 2023, Dong 2018, Xie 2019, Carlini et al. 2024) to evaluate and improve multimodal robustness.

- Do not use these techniques against systems you do not own or are not authorized to test.
- Embedded target strings are drawn from published academic evaluation sets.
- All four defenses in §3 should be evaluated end-to-end before any deployment decision.

---

### What's in this notebook

- **§1** — installs, repo clone, model pre-download. Same idempotent mechanism as Ch1/Ch2.
- **§2** — two attack scopes:
  - **§2.1** Classifier attacks (FGSM/PGD/CW/DeepFool/SmoothFool) on torchvision ResNet50 / InceptionV3.
  - **§2.2** VLM PGD with four loss choices, mirroring `qbtrain/cursedpixels`.
- **§3** — four defenses (randomized smoothing, input transforms, multi-view voting, feature squeeze).
- **§4** — variants: Universal Adversarial Perturbations, embedding-space attack, MI-FGSM, DI-FGSM, GCG single-token.


## 1. Setup

Same idempotent install/clone/prefetch mechanism as Ch1/Ch2. Re-runs detect what's already present and skip. The first cell may trigger a kernel restart if new packages were installed; if it does, re-run §1 from the top.


### 1.1 Install Python dependencies


In [1]:
import importlib, subprocess, sys, os

REQUIRED = [
    # Ch1 / Ch2 base set
    ('torch',          'torch'),
    ('transformers',   'transformers>=4.45'),
    ('accelerate',     'accelerate'),
    ('PIL',            'pillow'),
    ('numpy',          'numpy<2.0'),
    ('matplotlib',     'matplotlib'),
    ('huggingface_hub','huggingface_hub'),
    ('requests',       'requests'),
    ('sentencepiece',  'sentencepiece'),
    ('protobuf',       'protobuf'),
    ('cv2',            'opencv-python'),
    ('skimage',        'scikit-image'),
    # Ch3 additions
    ('torchvision',    'torchvision'),
]

def is_installed(modname, pip_name):
    try:
        importlib.import_module(modname)
        return True
    except ImportError:
        pass
    # Fallback for packages whose import name != pip name (e.g. protobuf →
    # google.protobuf). Ask pip directly.
    try:
        pkg = pip_name.split('>')[0].split('=')[0].split('<')[0].strip()
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', pkg],
            capture_output=True, text=True, timeout=10,
        )
        return result.returncode == 0
    except Exception:
        return False

to_install = [pip_name for mod, pip_name in REQUIRED if not is_installed(mod, pip_name)]

if to_install:
    print(f'Installing {len(to_install)} missing packages: {to_install}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + to_install)
    print('\nInstall complete. Restarting kernel — please re-run this cell after restart.')
    os.kill(os.getpid(), 9)
else:
    print('All Python dependencies already installed. No restart needed.')

# Windows + CUDA workaround: pyarrow (used by HF `datasets`) and torch can
# segfault when torch loads first. Eager-import `datasets` here so all
# subsequent cells (which import torch) are safe. No-op on systems where
# the segfault does not occur.
try:
    import datasets  # noqa: F401
except Exception:
    pass


All Python dependencies already installed. No restart needed.


### 1.2 Clone the cvpr-tutorial-repo

Same repo as Ch1/Ch2. If you ran either previously, this finds the existing clone and skips.


In [2]:
import os, sys, subprocess

REPO_URL = 'https://github.com/pavanreddyml/cvpr-tutorial-repo.git'

CANDIDATES = [
    os.path.abspath(os.path.join(os.getcwd(), 'cvpr-tutorial-repo')),
    os.path.abspath(os.path.join(os.getcwd(), '..', 'cvpr-tutorial-repo')),
    '/content/cvpr-tutorial-repo',
]

REPO_DIR = None
for cand in CANDIDATES:
    if os.path.isdir(cand) and os.path.isfile(os.path.join(cand, 'ch3', '__init__.py')):
        REPO_DIR = cand
        print(f'Found existing repo at: {REPO_DIR}')
        break

if REPO_DIR is None:
    REPO_DIR = CANDIDATES[0]
    print(f'Cloning {REPO_URL} → {REPO_DIR}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import ch1, ch3
print(f'ch1 v{ch1.__version__} loaded from: {ch1.__file__}')
print(f'ch3 v{ch3.__version__} loaded from: {ch3.__file__}')


Found existing repo at: /content/cvpr-tutorial-repo
ch1 v0.1.0 loaded from: /content/cvpr-tutorial-repo/ch1/__init__.py
ch3 v0.1.0 loaded from: /content/cvpr-tutorial-repo/ch3/__init__.py


### 1.3 Pre-download models + sample images

- SmolVLM-Instruct (~5GB, may be cached from Ch1/Ch2) — for §2.2 + §4
- ResNet50 + InceptionV3 torchvision weights (~100MB each) — for §2.1 + §3 + §4 classifier attacks
- 4 sample ImageNet photos


In [ ]:
from huggingface_hub import snapshot_download
from ch3.samples import prefetch_all
import time

MODELS_TO_PREFETCH = [
    ('HuggingFaceTB/SmolVLM-Instruct', 'VLM (default for §2.2/§4)'),
]

import threading

IGNORE = ['*.bin', '*.h5', '*.msgpack', '*.gguf', 'onnx/*', '*.onnx', '*.onnx_data']

def _fetch(repo_id):
    return snapshot_download(repo_id=repo_id, ignore_patterns=IGNORE)

first_repo, first_label = MODELS_TO_PREFETCH[0]
print(f'⬇  Downloading first model SYNCHRONOUSLY: {first_repo}')
print(f'   — {first_label}')
t0 = time.time()
try:
    _fetch(first_repo)
    elapsed = time.time() - t0
    note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
    print(f'   ✅ ready {note}')
except Exception as e:
    print(f'   ❌ FAILED: {e}')

if len(MODELS_TO_PREFETCH) > 1:
    def _bg(repo_id, label):
        t0 = time.time()
        try:
            _fetch(repo_id)
            elapsed = time.time() - t0
            note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
            print(f'\n   ✅ [bg] {repo_id} ready {note} — {label}')
        except Exception as e:
            print(f'\n   ❌ [bg] {repo_id} FAILED: {e}')
    print(f'\n⏳ Starting {len(MODELS_TO_PREFETCH) - 1} background download(s)...')
    for repo_id, label in MODELS_TO_PREFETCH[1:]:
        threading.Thread(target=_bg, args=(repo_id, label), daemon=True).start()

print('\n=== torchvision ImageNet classifiers ===')
import torchvision.models as tvm
for short, attr, weights_attr, size in [
    ('resnet50', 'resnet50', 'ResNet50_Weights', 224),
    ('inception_v3', 'inception_v3', 'Inception_V3_Weights', 299),
]:
    t0 = time.time()
    try:
        weights = getattr(tvm, weights_attr).DEFAULT
        # Trigger download by instantiating the model
        if short == 'inception_v3':
            m = getattr(tvm, attr)(weights=weights, aux_logits=True, init_weights=False)
        else:
            m = getattr(tvm, attr)(weights=weights)
        del m
        elapsed = time.time() - t0
        cached = '(cached)' if elapsed < 3 else f'({elapsed:.1f}s)'
        print(f'  ✅ {short:20s} input={size}px  {cached}')
    except Exception as e:
        print(f'  ❌ {short:20s} FAILED: {e}')

print('\n=== Sample images ===')
for name, path in prefetch_all().items():
    print(f'  {name:20s} → {path}')

print('\nAll set. Move to §2.')


⬇  Downloading first model SYNCHRONOUSLY: HuggingFaceTB/SmolVLM-Instruct
   — VLM (default for §2.2/§4)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

### 1.4 GPU summary


In [ ]:
from ch1.models import gpu_status
print(gpu_status())


## 2. Cursed Pixels — Adversarial Perturbations

Two scopes:
- **§2.1** Classifier attacks. FGSM/PGD/CW/DeepFool/SmoothFool on a torchvision ImageNet model. Output: a class label. Clear loss, fast, classical first-order attacks.
- **§2.2** VLM PGD. Same idea, harder problem: output is free-form text. We adapt PGD to the VLM by computing the loss on logits (teacher forcing) and backproping through the vision encoder + projection + LLM all the way back to input pixels.


### 2.1 Classifier attacks (FGSM / PGD / CW / DeepFool / SmoothFool)

Run any of the 5 attacks on a torchvision ImageNet model. The dashboard updates every step, showing original / adversarial / perturbation / gradient / loss curve + top-5 predictions.


In [ ]:
#@title ⚙️ §2.1 Classifier Attack Configuration { run: "auto" }

#@markdown **Attack** — first-order attack family. CW and DeepFool/SmoothFool are slower.
ATTACK = "pgd" #@param ["fgsm", "pgd", "cw", "deepfool", "smoothfool"]

#@markdown **Model** — torchvision ImageNet classifier.
CLASSIFIER = "resnet50" #@param ["resnet50", "inception_v3"]

#@markdown **Sample image** — clean input to attack.
SAMPLE = "panda" #@param ["panda", "cat", "tiger", "golden_retriever"]

#@markdown **Targeted attack?** — if True, push toward TARGET_CLASS instead of away from the true class.
TARGETED = True #@param {type: "boolean"}
TARGET_CLASS = 20 #@param {type: "integer"}

#@markdown **ε (L∞ bound, n/255 units)** — only used by FGSM, PGD, MI-FGSM.
EPSILON_255 = 8 #@param {type: "slider", min: 1, max: 64, step: 1}
#@markdown **α (PGD step size, n/255 units)**.
ALPHA_255 = 2 #@param {type: "slider", min: 1, max: 16, step: 1}
#@markdown **Iterations** — PGD/CW/DeepFool steps.
NUM_STEPS = 40 #@param {type: "slider", min: 1, max: 200, step: 1}

#@markdown **C&W trade-off c, confidence κ, learning rate** (CW only).
CW_C = 1.0 #@param {type: "slider", min: 0.01, max: 10.0, step: 0.1}
CW_KAPPA = 0.0 #@param {type: "slider", min: 0.0, max: 50.0, step: 1.0}
CW_LR = 0.01 #@param {type: "slider", min: 0.001, max: 0.1, step: 0.001}

#@markdown **DeepFool / SmoothFool** overshoot + candidate-class count + Gaussian σ.
OVERSHOOT = 0.02 #@param {type: "slider", min: 0.0, max: 0.2, step: 0.01}
CANDIDATE_CLASSES = 10 #@param {type: "slider", min: 2, max: 20, step: 1}
SMOOTHFOOL_SIGMA = 1.0 #@param {type: "slider", min: 0.5, max: 3.0, step: 0.1}

#@markdown **Dashboard refresh interval (steps)** — lower = more frequent updates.
REFRESH_EVERY = 2 #@param {type: "slider", min: 1, max: 10, step: 1}

EPSILON = EPSILON_255 / 255.0
ALPHA = ALPHA_255 / 255.0
print(f'Attack         : {ATTACK}')
print(f'Classifier     : {CLASSIFIER}')
print(f'Sample         : {SAMPLE}')
print(f'Targeted       : {TARGETED}'
      + (f' (target class {TARGET_CLASS})' if TARGETED else ''))
print(f'ε / α          : {EPSILON_255}/255 / {ALPHA_255}/255')
print(f'Iterations     : {NUM_STEPS}')
print(f'Refresh every  : {REFRESH_EVERY} step(s)')


Clear GPU + load the chosen classifier. Run this whenever you change `CLASSIFIER`.


In [ ]:
from ch3 import classifier_models as cm

cm.unload()
print('GPU cleared:', cm.gpu_status())

print(f'\nLoading {CLASSIFIER}...')
clf = cm.load_classifier(CLASSIFIER)
print(f'✅ Loaded {CLASSIFIER}  input={clf.input_size}px  {len(clf.categories)} classes')
print('GPU now    :', cm.gpu_status())


Run the attack with a live dashboard updating every `REFRESH_EVERY` steps. The dashboard shows original / adversarial / perturbation / gradient / loss curve + top-5 predictions with the true label highlighted in green.


In [ ]:
from ch3 import classifier_attacks as ca
from ch3 import render as r3
from ch3.samples import load_sample
from IPython.display import display, HTML, clear_output

pil = load_sample(SAMPLE)

# Build attack-specific params
params = {}
if ATTACK == 'fgsm':
    params = dict(epsilon=EPSILON, targeted=TARGETED, target_class=TARGET_CLASS)
elif ATTACK == 'pgd':
    params = dict(epsilon=EPSILON, alpha=ALPHA, num_steps=NUM_STEPS,
                   targeted=TARGETED, target_class=TARGET_CLASS, early_stop=True)
elif ATTACK == 'cw':
    params = dict(num_steps=NUM_STEPS, c=CW_C, kappa=CW_KAPPA, lr=CW_LR,
                   targeted=TARGETED, target_class=TARGET_CLASS, early_stop=True)
elif ATTACK == 'deepfool':
    params = dict(num_steps=NUM_STEPS, overshoot=OVERSHOOT,
                   num_candidate_classes=CANDIDATE_CLASSES, sigma=0.0, early_stop=True)
elif ATTACK == 'smoothfool':
    params = dict(num_steps=NUM_STEPS, overshoot=OVERSHOOT,
                   num_candidate_classes=CANDIDATE_CLASSES, sigma=SMOOTHFOOL_SIGMA, early_stop=True)

target_label = (clf.categories[TARGET_CLASS]
                if TARGETED and TARGET_CLASS < len(clf.categories) else None)

losses = []
last_frame = None
step = 0
for frame in ca.run_attack(ATTACK, clf, pil, **params):
    step = frame['step']
    if frame.get('loss') is not None:
        losses.append(frame['loss'])
    last_frame = frame
    if step % REFRESH_EVERY == 0 or step == 1:
        clear_output(wait=True)
        display(HTML(r3.live_classifier_dashboard(
            frame, losses,
            attack_name=ATTACK, model_name=CLASSIFIER,
            num_steps=NUM_STEPS if ATTACK != 'fgsm' else 1,
            epsilon=EPSILON if ATTACK in ('fgsm', 'pgd') else None,
            targeted=TARGETED, target_label=target_label,
        )))

# Final render
if last_frame is not None:
    clear_output(wait=True)
    display(HTML(r3.live_classifier_dashboard(
        last_frame, losses, attack_name=ATTACK, model_name=CLASSIFIER,
        num_steps=last_frame['step'],
        epsilon=EPSILON if ATTACK in ('fgsm', 'pgd') else None,
        targeted=TARGETED, target_label=target_label,
    )))
    display(HTML(r3.render_classifier_final(
        attack_name=ATTACK, model_name=CLASSIFIER,
        orig_np=last_frame['x_orig_np'],
        adv_np=last_frame['x_adv_np'],
        delta_np=last_frame['delta_vis_np'],
        orig_label=last_frame['orig_label'],
        adv_label=last_frame['top1_label'],
        orig_prob=last_frame['orig_prob'],
        adv_prob=last_frame['top1_prob'],
        linf=last_frame['linf'], l2=last_frame['l2'],
        psnr=last_frame.get('psnr'),
        losses=losses,
    )))

# Export the (clean, adversarial) pair so §3 can reuse it instead of
# regenerating the attack from scratch.
from PIL import Image as _Image
clean_pil = pil
adv_pil = _Image.fromarray((last_frame['x_adv_np'] * 255).clip(0, 255).astype('uint8'))
print(f'\nExported: clean_pil  ({clean_pil.size})  '
      f'+ adv_pil  ({adv_pil.size})  -> §3 will reuse these.')


### 2.2 VLM PGD — adapting first-order attacks to vision-language models

Same PGD machinery, but now the loss operates on the LLM's logits (teacher forcing) and the backward pass flows all the way through the LLM, the multimodal projection, and the vision encoder back to the input pixel values. Choose one of four loss functions in the params.

Ported from `qbtrain/apps/aisecurity/cursedpixels/functions.py`. Same dashboard style as the existing `cvpr/1 - cursed_pixels.ipynb`.


In [ ]:
#@title ⚙️ §2.2 VLM PGD Configuration { run: "auto" }

#@markdown **Target preset** — what the attacker wants the VLM to say.
#@markdown Mix of harmful (Qi 2024 academic eval set) and milder unethical examples.
TARGET_PRESET = "phishing_email" #@param ["phishing_email", "exfil_memory", "fake_excuse", "exam_cheat", "advertisement", "wrong_animal"]

#@markdown **Loss function** — see slide 13-17. CE is most reliable; GCG is 10× faster but less so.
VLM_LOSS = "target_token_ce" #@param ["target_token_ce", "refusal_suppression", "logit_margin", "gcg_single_token"]

#@markdown **VLM** — open-source model with public weights (white-box attacker scenario).
VLM_MODEL_ID = "smolvlm" #@param ["smolvlm", "smolvlm-256m", "llava-1.5-7b", "qwen2-vl-2b", "qwen2-vl-7b"]

#@markdown **Sample image** — clean cover.
VLM_SAMPLE = "panda" #@param ["panda", "cat", "tiger", "golden_retriever"]

#@markdown **ε (raw n/255)** — perturbation budget. 16/255 ≈ 6% per pixel, invisible.
VLM_EPSILON_255 = 16 #@param {type: "slider", min: 1, max: 64, step: 1}
#@markdown **α (raw n/255)** — PGD step size.
VLM_STEP_255 = 1 #@param {type: "slider", min: 1, max: 8, step: 1}
#@markdown **Steps** — typically 100-300 for full convergence. Set lower for faster demos.
VLM_NUM_STEPS = 200 #@param {type: "slider", min: 10, max: 500, step: 10}
#@markdown **Snapshot every N steps** — re-renders dashboard + runs generation.
VLM_EVAL_EVERY = 10 #@param {type: "slider", min: 2, max: 50, step: 1}
#@markdown **Max output tokens per generation** — length of each snapshot/final
#@markdown VLM completion shown in the dashboard. Default 50 truncates many
#@markdown target outputs mid-sentence; raise to ~200 to see the full attacker
#@markdown payload. Higher values slow each snapshot linearly (one generate()
#@markdown call per snapshot, plus one at init for the clean baseline).
VLM_GEN_MAX_TOKENS = 200 #@param {type: "slider", min: 20, max: 400, step: 20}

from ch3.scenarios import get_vlm_target, ETHICS_NOTICE
tgt = get_vlm_target(TARGET_PRESET)
print(f'Target         : {tgt.label}  ({tgt.category})')
print(f'Prompt         : {tgt.prompt}')
print(f'Target text    : {tgt.target_text}')
print(f'Loss           : {VLM_LOSS}')
print(f'VLM            : {VLM_MODEL_ID}')
print(f'Sample         : {VLM_SAMPLE}')
print(f'ε raw          : {VLM_EPSILON_255}/255')
print(f'α raw          : {VLM_STEP_255}/255')
print(f'Steps          : {VLM_NUM_STEPS}  (snapshot every {VLM_EVAL_EVERY})')
print(f'Gen max tokens : {VLM_GEN_MAX_TOKENS}')
print(f'\n{ETHICS_NOTICE}')


Clear GPU + load the chosen VLM. Run whenever `VLM_MODEL_ID` changes (or after running a §2.1 classifier cell which loaded a different model into GPU).


In [ ]:
from ch1 import models as ch1_models
from ch3 import classifier_models as cm

cm.unload()       # release ResNet/Inception from §2.1
ch1_models.unload()
print('GPU cleared:', ch1_models.gpu_status())

print(f'\nLoading VLM {VLM_MODEL_ID}...')
vlm = ch1_models.load_vlm(VLM_MODEL_ID)
print(f'✅ Loaded {vlm.hf_id}')
print('GPU now    :', ch1_models.gpu_status())


Run the VLM PGD attack with a live dashboard. Snapshots include the current generation from the perturbed image so you can watch the model's output drift toward the target.


In [ ]:
from ch3 import vlm_pgd
from ch3 import render as r3
from ch3.samples import load_sample
from IPython.display import display, HTML, clear_output
import numpy as np

pil_sample = load_sample(VLM_SAMPLE)

losses = []
baseline_response = None
original_np = None
last_snapshot = None
best_loss = float('inf')
best_step = 0
final_event = None

for event in vlm_pgd.vlm_pgd_attack(
    vlm, pil_sample,
    prompt=tgt.prompt, target_text=tgt.target_text,
    loss_function=VLM_LOSS,
    epsilon_raw=VLM_EPSILON_255 / 255.0,
    step_size_raw=VLM_STEP_255 / 255.0,
    num_steps=VLM_NUM_STEPS,
    eval_every=VLM_EVAL_EVERY,
    gen_max_new_tokens=VLM_GEN_MAX_TOKENS,
):
    et = event['type']
    if et == 'init':
        baseline_response = event['baseline_response']
        original_np = event['original_image_np']
    elif et == 'loss':
        losses.append(event['loss'])
        if event['best_loss'] < best_loss:
            best_loss = event['best_loss']
            best_step = event['step']
    elif et == 'snapshot':
        last_snapshot = event
        # Add original (clean) image for the dashboard
        last_snapshot['_orig_np'] = original_np
        last_snapshot['original_image_np'] = original_np
        clear_output(wait=True)
        display(HTML(r3.live_vlm_dashboard(
            last_snapshot, losses,
            model_name=vlm.hf_id, num_steps=VLM_NUM_STEPS,
            epsilon=event.get('linf', 0) and (VLM_EPSILON_255 / 255.0) * 2,
            loss_function=VLM_LOSS,
            prompt=tgt.prompt, target_text=tgt.target_text,
        )))
    elif et == 'done':
        final_event = event

# Final result
if final_event:
    clear_output(wait=True)
    display(HTML(r3.render_vlm_final(
        model_name=vlm.hf_id,
        target_label=tgt.label,
        prompt=tgt.prompt,
        target_text=tgt.target_text,
        baseline_response=baseline_response or '—',
        final_response=final_event['final_response'],
        original_np=original_np,
        adv_np=final_event['final_image_np'],
        noise_np=final_event['final_noise_np'],
        best_loss=final_event['best_loss'],
        linf=final_event['linf'],
        psnr=final_event.get('psnr'),
    )))
    print(f'\nBest loss reached at step {best_step}: {best_loss:.4f}')

    # Export the (clean, adversarial) pair so §3 reuses THIS attack's adv.
    # This overwrites whatever §2.1 exported — running §2.2 last means §3
    # sees the VLM-targeted adv (the one carrying the phishing payload).
    # Re-run §2.1 if you want the classifier-targeted adv back.
    from PIL import Image as _Image
    clean_pil = pil_sample
    adv_pil = _Image.fromarray((final_event['final_image_np'] * 255).clip(0, 255).astype('uint8'))
    print(f'Exported: clean_pil ({clean_pil.size}) + adv_pil ({adv_pil.size}) '
          f'-> §3 will reuse this VLM-targeted adv.')


## 3. Defenses

Four defenses, all evaluated against the classifier attacks from §2.1 (clean classifier outputs make the with/without comparison crisp). Per slide 28's eval table:

| # | Defense                    | ASR drop      | Cost          | Slide  |
|---|----------------------------|---------------|---------------|--------|
| 1 | Randomized smoothing       | 92% → 34%     | +10× compute  | D1     |
| 2 | Input transforms (chained) | 92% → 35%     | +20ms         | D2     |
| 3 | Multi-view voting          | 92% → 28%     | +5× compute   | D3     |
| 4 | Feature squeeze (detection)| —             | +2× compute   | D5     |

**No single defense gets ASR below 10%.** Unlike Ch1/Ch2, this is the hardest defense problem in VLM security — adversarial perturbations exploit non-robust features in the vision encoder itself. Best practice: stack 2-3 of these.

For each defense we re-use the adversarial image from §2.1's last attack (or recompute a quick PGD attack if you skipped §2.1).


First, pick the **probe model** — classifier or VLM. Both the classifier and the VLM probe operate on the `(clean_pil, adv_pil)` pair exported by §2.1, so re-running §2.1 with a different scenario / attack feeds straight into the defenses below. If §2.1 hasn't been run yet, this cell falls back to a quick inline PGD (vs ResNet50, ε=8/255, 30 steps) so §3 still stands alone.


In [ ]:
from ch3 import classifier_models as cm
#@title ⚙️ §3 — Probe model { run: "auto" }
#@markdown The defenses below show **BEFORE vs AFTER** the defense is applied.
#@markdown Pick which model interprets the (clean, adversarial) images:
#@markdown - **resnet50 / inception_v3** — ImageNet classifiers. BEFORE/AFTER
#@markdown   shows the top-1 label, so you can see if the defense **recovered**
#@markdown   the clean prediction.
#@markdown - **a VLM** (qwen2-vl-2b etc.) — BEFORE/AFTER shows the model's
#@markdown   generated description. Useful for seeing if the perturbation
#@markdown   transfers to a VLM and whether the defense's image transformation
#@markdown   changes the VLM's reading.
#@markdown Note: even in VLM mode, the adversarial image itself is always
#@markdown crafted against ResNet50 (the attack target) — the probe is just
#@markdown how we *interpret* the (clean, adv) pair.
PROBE_MODEL = "smolvlm" #@param ["resnet50", "inception_v3", "qwen2-vl-2b", "qwen2-vl-7b", "llava-1.5-7b", "smolvlm", "smolvlm-256m"]
#@markdown **VLM prompt** — what the user asks. Only used when PROBE_MODEL is a VLM.
#@markdown Avoid "answer briefly" — small VLMs take it literally and emit
#@markdown a one-word response ("Panda."), which makes the BEFORE/ATTACKED
#@markdown comparison hard to read. Default below asks for a description.
PROBE_VLM_PROMPT = "Describe this image." #@param {type: "string"}
#@markdown **VLM max-new-tokens** — length of each VLM response. 150-200 gives
#@markdown a 2-3 sentence description on smolvlm / qwen2-vl-2b.
PROBE_VLM_MAX_TOKENS = 250 #@param {type: "slider", min: 30, max: 400, step: 10}

PROBE_KIND = 'classifier' if PROBE_MODEL in ('resnet50', 'inception_v3') else 'vlm'
print(f'PROBE_MODEL = {PROBE_MODEL}  (kind={PROBE_KIND})')

# --- Stage 1: reuse the adversarial pair from §2.1 if available ---
# §2.1's attack cell exports `clean_pil` and `adv_pil` at the end. The
# defenses operate on whatever was crafted up there — same SAMPLE, same
# CLASSIFIER, same ε/α/steps. If §2.1 wasn't run yet we fall back to a
# quick inline PGD (vs ResNet50) so the cell is still self-contained.
from ch3 import classifier_attacks as ca
from ch3.samples import load_sample
from PIL import Image
import numpy as np
import torch

if 'clean_pil' in globals() and 'adv_pil' in globals():
    print(f'Reusing (clean_pil, adv_pil) exported by the most recent §2 cell.  '
          f'clean={clean_pil.size}  adv={adv_pil.size}')
    print('   (re-run §2.1 to use the classifier-targeted adv; '
          're-run §2.2 to use the VLM-targeted adv)')
else:
    print('No §2 cell has been run yet — generating an inline adversarial '
          '(PGD vs ResNet50, 30 steps) so §3 can stand alone.')
    if cm.current() is None or cm.current().short_id != 'resnet50':
        cm.unload()
        cm.load_classifier('resnet50')
    _attack_clf = cm.current()
    clean_pil = load_sample(SAMPLE)
    frames = list(ca.pgd_attack(_attack_clf, clean_pil, epsilon=8/255,
                                  alpha=2/255, num_steps=30, early_stop=True))
    last = frames[-1]
    adv_pil = Image.fromarray((last['x_adv_np'] * 255).clip(0, 255).astype('uint8'))
    print(f'  flipped {last["orig_label"]} → {last["top1_label"]} '
          f'(prob {last["top1_prob"]:.1%})')

# --- Stage 2: load the chosen PROBE and build a uniform probe() func ---
# `probe(pil)` always returns a short display string (label+prob OR
# generated text), so defense cells can render BEFORE/AFTER uniformly.
if PROBE_KIND == 'classifier':
    if cm.current() is None or cm.current().short_id != PROBE_MODEL:
        cm.unload()
        cm.load_classifier(PROBE_MODEL)
    clf = cm.current()
    _forward = cm.make_forward(clf)
    def probe(pil_img):
        x = cm.pil_to_tensor_01(pil_img.convert('RGB'),
                                  clf.input_size).to(clf.device)
        p = cm.predict_class(_forward, x, clf.categories)
        return f"{p['label']}  ({p['prob']:.1%})"
else:
    # VLM probe — unload the classifier (single-GPU memory budget).
    from ch1 import models as ch1_models
    cm.unload()
    if (ch1_models.current() is None or
        ch1_models.current().short_id != PROBE_MODEL):
        ch1_models.unload()
        print(f'Loading VLM {PROBE_MODEL}...')
        ch1_models.load_vlm(PROBE_MODEL)
    print(f'VLM ready: {ch1_models.current().hf_id}')
    clf = None  # marker that the defense cells must NOT call classifier defenses
    def probe(pil_img):
        return ch1_models.generate(
            prompt=PROBE_VLM_PROMPT, image=pil_img,
            max_new_tokens=PROBE_VLM_MAX_TOKENS,
        )

# Pre-defense (BEFORE) responses — shown in every defense panel header.
clean_resp_before = probe(clean_pil)
adv_resp_before   = probe(adv_pil)
print(f'  CLEAN before defense: {clean_resp_before[:90]}')
print(f'  ADV   before defense: {adv_resp_before[:90]}')


### 3.1 Defense 1 — Randomized Smoothing

Add K Gaussian noise samples to the image, predict each, aggregate. Cohen et al. 2019 established that this provides a **certified radius** within which no perturbation can change the output. Trade-off: noise also degrades clean accuracy by 3-8%.


In [ ]:
#@title ⚙️ Defense 3.1 — Randomized Smoothing { run: "auto" }
SMOOTH_SIGMA = 0.25 #@param {type: "slider", min: 0.05, max: 1.0, step: 0.05}
SMOOTH_K = 10 #@param {type: "slider", min: 1, max: 50, step: 1}
SMOOTH_SEED = 0 #@param {type: "integer"}
print(f'sigma={SMOOTH_SIGMA}, K={SMOOTH_K}, seed={SMOOTH_SEED}')


In [ ]:
from ch3 import defenses as ch3d
from ch3 import render as r3
from IPython.display import HTML

if PROBE_KIND == 'classifier':
    result = ch3d.run_randomized_smoothing_defense(
        clf=clf, clean_image=clean_pil, adv_image=adv_pil,
        sigma=SMOOTH_SIGMA, K=SMOOTH_K, seed=SMOOTH_SEED,
    )
    clean_p = result['clean_prediction']; adv_p = result['adv_prediction']
    clean_resp_after = f"{clean_p['label']}  ({clean_p['prob']:.1%})"
    adv_resp_after   = f"{adv_p['label']}  ({adv_p['prob']:.1%})"
    clean_status = (f'aggregated prob: {clean_p["prob"]:.1%}\n'
                    f'individual votes: {clean_p["votes"]}')
    adv_status = (f'aggregated prob: {adv_p["prob"]:.1%}\n'
                  f'individual votes: {adv_p["votes"]}')
    recovered = (adv_p['idx'] == clean_p['idx'])
    clean_img_show, adv_img_show = clean_pil, adv_pil
else:
    # VLM mode: smoothing aggregates K classifier predictions; for a VLM we
    # just show how a single Gaussian-noised version of the image reads.
    rng = np.random.default_rng(SMOOTH_SEED)
    def _noisy(pil):
        arr = np.asarray(pil.convert('RGB'), dtype=np.float32) / 255.0
        out = np.clip(arr + rng.normal(0, SMOOTH_SIGMA, arr.shape), 0, 1)
        return Image.fromarray((out * 255).astype(np.uint8))
    clean_img_show = _noisy(clean_pil); adv_img_show = _noisy(adv_pil)
    clean_resp_after = probe(clean_img_show)
    adv_resp_after   = probe(adv_img_show)
    clean_status = f'VLM probe on σ={SMOOTH_SIGMA} noised image (one sample of K)'
    adv_status   = f'VLM probe on σ={SMOOTH_SIGMA} noised image (one sample of K)'
    recovered = (clean_resp_after.lower() == adv_resp_after.lower())  # rough proxy

HTML(r3.render_defense_three_panel(
    defense_name='Randomized Smoothing',
    description=(
        f'Adds Gaussian noise (σ={SMOOTH_SIGMA}) to the input K={SMOOTH_K} times and '
        'aggregates the classifier softmax — the certified radius r is proportional to σ. '
        'In VLM mode we show ONE noised sample of the adversarial image (smoothing per se '
        "isn't applicable to a generative model; this is just to see how σ-level noise "
        'affects the VLM\u2019s reading).'
    ),
    badges=[('probe', PROBE_MODEL), ('sigma', SMOOTH_SIGMA), ('K', SMOOTH_K),
             ('recovered?', 'YES' if recovered else 'NO')],
    clean_image=clean_pil,    clean_response=clean_resp_before,
    adv_image=adv_pil,        adv_response=adv_resp_before,
    defended_image=adv_img_show, defended_response=adv_resp_after,
    recovered=recovered,
))


### 3.2 Defense 2 — Input Transformations (chained)

JPEG re-encode → bit reduction → Gaussian blur → random resize. Each transform is lossy enough to destroy the adversarial perturbation while leaving the semantic content intact. Cheap (~20ms total). Drops ASR ~92% → 35% per the slide.


In [ ]:
#@title ⚙️ Defense 3.2 — Input Transforms { run: "auto" }
INPUT_JPEG_Q = 75 #@param {type: "slider", min: 30, max: 95, step: 5}
INPUT_BITS = 4 #@param {type: "slider", min: 1, max: 8, step: 1}
INPUT_BLUR = 1.0 #@param {type: "slider", min: 0.0, max: 3.0, step: 0.25}
INPUT_RESIZE_LOW = 200 #@param {type: "slider", min: 128, max: 280, step: 8}
INPUT_RESIZE_HIGH = 248 #@param {type: "slider", min: 160, max: 320, step: 8}
INPUT_SEED = 0 #@param {type: "integer"}
print(f'JPEG q={INPUT_JPEG_Q} | bits={INPUT_BITS} | blur σ={INPUT_BLUR} | '
      f'resize ∈ [{INPUT_RESIZE_LOW}, {INPUT_RESIZE_HIGH}] | seed={INPUT_SEED}')


In [ ]:
from ch3 import defenses as ch3d
from ch3 import render as r3
from IPython.display import HTML

if PROBE_KIND == 'classifier':
    result = ch3d.run_input_transform_defense(
        clf=clf, clean_image=clean_pil, adv_image=adv_pil,
        jpeg_q=INPUT_JPEG_Q, bits=INPUT_BITS, blur_radius=INPUT_BLUR,
        resize_low=INPUT_RESIZE_LOW, resize_high=INPUT_RESIZE_HIGH, seed=INPUT_SEED,
    )
    clean_p = result['clean_prediction']; adv_p = result['adv_prediction']
    clean_resp_after = f"{clean_p['label']}  ({clean_p['prob']:.1%})"
    adv_resp_after   = f"{adv_p['label']}  ({adv_p['prob']:.1%})"
    clean_img_show = result['clean_transformed_image']
    adv_img_show   = result['adv_transformed_image']
    recovered = (adv_p['idx'] == clean_p['idx'])
else:
    # VLM mode: apply the same chained transform, probe the VLM on the result.
    def _transform(pil):
        return ch3d.chained_transform(
            pil, jpeg_q=INPUT_JPEG_Q, bits=INPUT_BITS, blur_radius=INPUT_BLUR,
            resize_low=INPUT_RESIZE_LOW, resize_high=INPUT_RESIZE_HIGH, seed=INPUT_SEED,
        )
    clean_img_show = _transform(clean_pil); adv_img_show = _transform(adv_pil)
    clean_resp_after = probe(clean_img_show)
    adv_resp_after   = probe(adv_img_show)
    recovered = (clean_resp_after.lower() == adv_resp_after.lower())

HTML(r3.render_defense_three_panel(
    defense_name='Input Transformations (chained)',
    description=(
        'JPEG re-encode → bit reduction → Gaussian blur → random resize. Each transform is '
        'lossy enough to destroy the adversarial perturbation while semantic content '
        'survives. ~20ms total. Drops classifier ASR ~92% → 35%. Column 3 shows the '
        'adversarial image AFTER the same transformations, probed by the chosen model.'
    ),
    badges=[('probe', PROBE_MODEL), ('jpeg q', INPUT_JPEG_Q), ('bits', INPUT_BITS),
             ('blur', INPUT_BLUR), ('recovered?', 'YES' if recovered else 'NO')],
    clean_image=clean_pil,    clean_response=clean_resp_before,
    adv_image=adv_pil,        adv_response=adv_resp_before,
    defended_image=adv_img_show, defended_response=adv_resp_after,
    recovered=recovered,
))


### 3.3 Defense 3 — Multi-view voting (with ABSTAIN)

Run K random augmentations (flip, rotate, JPEG, blur). If agreement < threshold → ABSTAIN. Adversarial N is optimized for one pixel layout; augmentations break it. Catches ~80% of adversarial images at K=5. Cost: 5× inference.


In [ ]:
#@title ⚙️ Defense 3.3 — Multi-view voting { run: "auto" }
MV_K = 5 #@param {type: "slider", min: 2, max: 10, step: 1}
MV_ABSTAIN_THRESHOLD = 0.6 #@param {type: "slider", min: 0.2, max: 0.95, step: 0.05}
MV_SEED = 0 #@param {type: "integer"}
print(f'K={MV_K} | abstain<{MV_ABSTAIN_THRESHOLD} | seed={MV_SEED}')


In [ ]:
from ch3 import defenses as ch3d
from ch3 import render as r3
from IPython.display import HTML
import random as _rnd

if PROBE_KIND == 'classifier':
    result = ch3d.run_multi_view_defense(
        clf=clf, clean_image=clean_pil, adv_image=adv_pil,
        K=MV_K, seed=MV_SEED, abstain_threshold=MV_ABSTAIN_THRESHOLD,
    )
    clean_v = result['clean_prediction']; adv_v = result['adv_prediction']
    clean_resp_after = (f'{clean_v["top_label"]}  '
                         f'(agreement {clean_v["agreement"]:.0%}'
                         + (', ABSTAIN' if clean_v['abstain'] else '') + ')')
    adv_resp_after = (f'{adv_v["top_label"]}  '
                       f'(agreement {adv_v["agreement"]:.0%}'
                       + (', ABSTAIN — flagged' if adv_v['abstain'] else '') + ')')
    clean_status = (f'votes: {clean_v["votes"]}\nabstain={clean_v["abstain"]}')
    adv_status   = (f'votes: {adv_v["votes"]}\nabstain={adv_v["abstain"]}')
    success_adv = adv_v['abstain']  # detection-style; abstaining IS the win
    clean_img_show, adv_img_show = clean_pil, adv_pil
else:
    # VLM mode: pick one augmentation from the same pool, apply, probe.
    rng = _rnd.Random(MV_SEED)
    aug = rng.choice(ch3d._augmentations(rng))
    clean_img_show = aug(clean_pil); adv_img_show = aug(adv_pil)
    clean_resp_after = probe(clean_img_show)
    adv_resp_after   = probe(adv_img_show)
    clean_status = 'VLM probe on one random augmentation (sample of K)'
    adv_status   = 'VLM probe on one random augmentation (sample of K)'
    success_adv = (clean_resp_after.lower() == adv_resp_after.lower())

HTML(r3.render_defense_three_panel(
    defense_name='Multi-view voting',
    description=(
        f'Apply K={MV_K} random augmentations (flip, rotate, JPEG, blur), predict each, '
        'and abstain if top-class agreement < threshold. Adversarial perturbations are '
        'optimized for one pixel layout; augmentations break them. Column 3 shows one '
        'sampled augmentation of the adversarial image (representative of the K votes).'
    ),
    badges=[('probe', PROBE_MODEL), ('K', MV_K), ('threshold', MV_ABSTAIN_THRESHOLD),
             ('recovered?', 'YES' if success_adv else 'NO')],
    clean_image=clean_pil,    clean_response=clean_resp_before,
    adv_image=adv_pil,        adv_response=adv_resp_before,
    defended_image=adv_img_show, defended_response=adv_resp_after,
    recovered=success_adv,
))


### 3.4 Defense 4 — Feature Squeeze (Xu et al. 2018) — detection

Squeeze the input two ways (bit reduction + Gaussian blur). If the model's softmax changes by more than threshold (L1 distance), flag as adversarial. This is a **detection** layer, not an active defense — it doesn't try to recover the clean prediction, just rejects suspicious inputs. Cheap, complements other defenses.


In [ ]:
#@title ⚙️ Defense 3.4 — Feature Squeeze { run: "auto" }
FS_BITS = 4 #@param {type: "slider", min: 1, max: 8, step: 1}
FS_BLUR = 1.0 #@param {type: "slider", min: 0.5, max: 3.0, step: 0.25}
FS_THRESHOLD = 1.2 #@param {type: "slider", min: 0.1, max: 2.0, step: 0.1}
print(f'bits={FS_BITS} | blur σ={FS_BLUR} | L1 threshold={FS_THRESHOLD}')


In [ ]:
from ch3 import defenses as ch3d
from ch3 import render as r3
from IPython.display import HTML

if PROBE_KIND == 'classifier':
    result = ch3d.run_feature_squeeze_defense(
        clf=clf, clean_image=clean_pil, adv_image=adv_pil,
        bits=FS_BITS, blur_radius=FS_BLUR, threshold=FS_THRESHOLD,
    )
    cr = result['clean_result']; ar = result['adv_result']
    clean_resp_after = (f'{cr["orig_label"]}  '
                         f'(max-L1 {cr["max_l1"]:.2f}'
                         + (', FLAGGED' if cr['flagged'] else '') + ')')
    adv_resp_after = (f'{ar["orig_label"]}  '
                       f'(max-L1 {ar["max_l1"]:.2f}'
                       + (', FLAGGED — likely adversarial' if ar['flagged'] else '') + ')')
    clean_status = f'bit-L1 {cr["bit_l1"]:.3f}, blur-L1 {cr["blur_l1"]:.3f}, thr {cr["threshold"]}'
    adv_status   = f'bit-L1 {ar["bit_l1"]:.3f}, blur-L1 {ar["blur_l1"]:.3f}, thr {ar["threshold"]}'
    success_adv = ar['flagged']  # detection: flagging IS success
    clean_img_show, adv_img_show = clean_pil, adv_pil
else:
    # VLM mode: apply the same squeeze (bit + blur) and probe.
    def _squeeze(pil):
        out = ch3d.bit_reduce(pil, bits=FS_BITS)
        return ch3d.gaussian_blur_pil(out, radius=FS_BLUR)
    clean_img_show = _squeeze(clean_pil); adv_img_show = _squeeze(adv_pil)
    clean_resp_after = probe(clean_img_show)
    adv_resp_after   = probe(adv_img_show)
    clean_status = f'VLM probe on bit-{FS_BITS} + blur σ={FS_BLUR}'
    adv_status   = f'VLM probe on bit-{FS_BITS} + blur σ={FS_BLUR}'
    success_adv = (clean_resp_after.lower() == adv_resp_after.lower())

HTML(r3.render_defense_three_panel(
    defense_name='Feature Squeeze (Xu et al. 2018)',
    description=(
        'Squeeze the input two ways (bit reduction + Gaussian blur). For the classifier, '
        'a large softmax-L1 distance between original and squeezed flags the image as '
        'adversarial (detection, not recovery). Column 3 shows the squeezed adversarial '
        "image, probed by the chosen model."
    ),
    badges=[('probe', PROBE_MODEL), ('bits', FS_BITS), ('blur', FS_BLUR),
             ('threshold', FS_THRESHOLD),
             ('recovered?', 'YES' if success_adv else 'NO')],
    clean_image=clean_pil,    clean_response=clean_resp_before,
    adv_image=adv_pil,        adv_response=adv_resp_before,
    defended_image=adv_img_show, defended_response=adv_resp_after,
    recovered=success_adv,
))


## 4. Variants — Other Pixel-Level Attacks

Four variants from the slides + adversarial-ML literature. Each runs end-to-end against the loaded models and prints a final HTML.

| Variant                                  | Key idea                                          | Reference                         |
|------------------------------------------|---------------------------------------------------|-----------------------------------|
| Universal Adversarial Perturbation       | One N for ANY image. Pre-compute once.            | Moosavi-Dezfooli 2017 (slide 18)  |
| Embedding-space attack                   | Match adv embedding to a target image's embedding | Carlini et al. 2024 (slide 15)    |
| MI-FGSM (momentum)                       | Better transferability across models              | Dong et al. 2018 (slide 19)       |
| DI-FGSM (input diversity + momentum)     | Best practice for attacking unknown models        | Xie et al. 2019 (slide 19)        |


### 4.1 Universal Adversarial Perturbation (UAP)

Slide 18, Moosavi-Dezfooli 2017. One perturbation N optimized over a batch of images, then applied to NEW images without re-running the optimizer. Lower per-image ASR but zero per-image cost.

We use the 4 sample images as the training batch, then test transfer to all 4.


In [ ]:
from ch3 import variants
from ch3 import classifier_models as cm
from ch3.samples import load_sample, list_samples
from ch3 import render as r3
from PIL import Image
import numpy as np
from IPython.display import display, HTML

# Ensure we have a classifier; reuse the one loaded earlier if compatible
if cm.current() is None:
    cm.load_classifier(CLASSIFIER)
clf = cm.current()

train_pils = [load_sample(n) for n in list_samples()]
print(f'Optimizing UAP over {len(train_pils)} images...')
uap = variants.universal_perturbation(
    clf, train_pils, epsilon=16/255, alpha=2/255, pgd_steps=10, num_passes=2,
)
print(f'  fool rate on training set: {uap["fool_rate"]:.0%}')
print(f'  v L∞ = {uap["v_linf"] * 255:.1f}/255   v L₂ = {uap["v_l2"]:.1f}')

# Apply to each sample
import torch
from ch3.classifier_models import make_forward, pil_to_tensor_01, predict_class
forward = make_forward(clf)

v_pil = Image.fromarray(((uap['v'].transpose(1, 2, 0) + 16/255) / (32/255) * 255).clip(0, 255).astype('uint8'))
columns = [{'label': 'UAP (rescaled view)', 'image': v_pil, 'caption': 'one tensor, every image'}]
extras_rows = []
for name in list_samples():
    pil = load_sample(name)
    x_clean = pil_to_tensor_01(pil, clf.input_size).to(clf.device)
    clean_pred = predict_class(forward, x_clean, clf.categories)
    adv_pil_uap = variants.apply_universal_perturbation(clf, pil, uap['v'])
    x_adv = pil_to_tensor_01(adv_pil_uap, clf.input_size).to(clf.device)
    adv_pred = predict_class(forward, x_adv, clf.categories)
    flipped = adv_pred['idx'] != clean_pred['idx']
    row_color = r3.COLORS['red'] if flipped else r3.COLORS['green']
    extras_rows.append(f'''
      <tr>
        <td style="padding:4px 8px; font-size:11px;">{name}</td>
        <td style="padding:4px 8px; font-size:11px;">{clean_pred['label']} ({clean_pred['prob']:.0%})</td>
        <td style="padding:4px 8px; font-size:11px; color:{row_color};">
          {adv_pred['label']} ({adv_pred['prob']:.0%}) {'❌ FLIPPED' if flipped else '✅'}
        </td>
      </tr>''')

extras_html = f'''
<div style="margin-top:14px; background:#161b22; padding:12px; border-radius:8px;">
  <div style="font-size:10px; color:#8b949e; text-transform:uppercase; letter-spacing:0.5px;
              margin-bottom:8px;">PER-IMAGE TRANSFER</div>
  <table style="font-family:monospace; border-collapse:collapse;">
    <tr style="color:#8b949e; font-size:10px;">
      <th style="padding:4px 8px; text-align:left;">image</th>
      <th style="padding:4px 8px; text-align:left;">clean</th>
      <th style="padding:4px 8px; text-align:left;">clean + UAP</th>
    </tr>
    {''.join(extras_rows)}
  </table>
</div>'''

display(HTML(r3.render_variant_panel(
    variant_name='Universal Adversarial Perturbation',
    paper_citation='Moosavi-Dezfooli 2017 (slide 18)',
    description=(
        f'One perturbation v optimized over {len(train_pils)} training images, applied '
        'to all 4 samples. Lower per-image ASR than per-image PGD, but zero compute cost '
        'to apply once computed. Useful for cheap mass attacks.'
    ),
    image_columns=columns,
    extras_html=extras_html,
)))


### 4.2 Embedding-space attack (Carlini et al. 2024)

Match the adversarial image's vision encoder embedding to a TARGET image's embedding. No LLM gradient needed — only the vision encoder. Transfers better across VLMs that share CLIP/SigLIP backbones.

Here: take the panda → optimize the panda so its embedding matches the tiger's.


In [ ]:
import torch
from PIL import Image
import numpy as np
from IPython.display import display, HTML

from ch1 import models as ch1_models
from ch3 import classifier_models as cm
from ch3 import variants
from ch3 import render as r3
from ch3.samples import load_sample

# 1. Free any classifier and load the small VLM.
TARGET_VLM = "smolvlm-256m"
cm.unload()
if not ch1_models.current() or ch1_models.current().short_id != TARGET_VLM:
    ch1_models.unload()
    print(f"Loading {TARGET_VLM}...")
    ch1_models.load_vlm(TARGET_VLM)
vlm = ch1_models.current()

# 2. Reset ve.forward to the TRUE original by deleting any instance-level
#    override, then re-patch. This survives any number of prior stale patches.
ve = vlm.model.model.vision_model
ve.__dict__.pop("forward", None)             # any prior monkey-patch gone
ve.__dict__.pop("_orig_forward_backup", None)
ve.__dict__.pop("_squeeze_patched", None)
_true_original = ve.forward                  # now the class's bound method
_patch_size = ve.config.patch_size
_target_dtype = ve.embeddings.patch_embedding.weight.dtype

def _patched_forward(pixel_values, patch_attention_mask=None, **kwargs):
    if pixel_values.dim() == 5:
        pixel_values = pixel_values.squeeze(1)
    pixel_values = pixel_values.to(_target_dtype)
    if patch_attention_mask is None:
        B, _, H, W = pixel_values.shape
        patch_attention_mask = torch.ones(
            (B, H // _patch_size, W // _patch_size),
            dtype=torch.bool, device=pixel_values.device,
        )
    return _true_original(
        pixel_values, patch_attention_mask=patch_attention_mask, **kwargs
    )
ve.forward = _patched_forward
print(f"Patched Idefics3 vision encoder.  target_dtype={_target_dtype}")

# 3. Run the attack.
source_pil = load_sample("panda")
target_pil = load_sample("tiger")

events = list(variants.embedding_attack(
    vlm, source_pil, target_pil,
    epsilon_raw=16 / 255, step_size_raw=1 / 255, num_steps=60,
))
init_evt = events[0]
done_evt = events[-1]
loss_evts = [e for e in events if e["type"] == "loss"]
print(f"best embedding loss: {done_evt['best_loss']:.4f}")
print(f"losses tracked     : {len(loss_evts)}")

# 4. VLM responses.
adv_pil_emb = Image.fromarray(
    (done_evt["final_image_np"] * 255).clip(0, 255).astype("uint8")
)
src_resp = ch1_models.generate(
    prompt="What animal is in this image?", image=source_pil, max_new_tokens=60,
)
tgt_resp = ch1_models.generate(
    prompt="What animal is in this image?", image=target_pil, max_new_tokens=60,
)
adv_resp = ch1_models.generate(
    prompt="What animal is in this image?", image=adv_pil_emb, max_new_tokens=60,
)

# 5. Render.
columns = [
    {"label": "SOURCE (panda)",  "image": source_pil,   "caption": src_resp[:120]},
    {"label": "TARGET (tiger)",  "image": target_pil,   "caption": tgt_resp[:120]},
    {"label": "ADVERSARIAL",     "image": adv_pil_emb,  "caption": adv_resp[:120]},
]
display(HTML(r3.render_variant_panel(
    variant_name="Embedding-space attack",
    paper_citation="Carlini et al. 2024 (slide 15)",
    description=(
        "Optimize source + δ so its vision encoder embedding matches the target. "
        "No LLM gradient needed — only the vision encoder. Transfers better "
        "across VLMs that share CLIP/SigLIP. Demo: push the panda toward the tiger."
    ),
    image_columns=columns,
)))


### 4.3 MI-FGSM (Dong et al. 2018)

Momentum-iterative FGSM. Accumulates gradient momentum across PGD steps — perturbations transfer much better across models (different architectures, different weights). Standard for attacking closed-source APIs.


In [ ]:
from ch3 import variants
from ch3 import classifier_models as cm
from ch3.samples import load_sample
from ch3 import render as r3
from PIL import Image
import numpy as np
from IPython.display import display, HTML
from ch1 import models as ch1_models

ch1_models.unload()
if cm.current() is None or cm.current().short_id != CLASSIFIER:
    cm.unload()
    cm.load_classifier(CLASSIFIER)
clf = cm.current()

pil = load_sample(SAMPLE)
events = list(variants.mi_fgsm_attack(clf, pil,
                                       epsilon=16/255, alpha=2/255,
                                       num_steps=20, mu=1.0))
init_evt = events[0]; done_evt = events[-1]
loss_history = [e['loss'] for e in events if e['type'] == 'loss']

adv_pil = Image.fromarray((done_evt['x_adv_np'] * 255).clip(0, 255).astype('uint8'))
delta_pil = Image.fromarray((done_evt['delta_np'] * 255).clip(0, 255).astype('uint8'))
orig_pil_view = Image.fromarray((done_evt['x_orig_np'] * 255).clip(0, 255).astype('uint8'))

extras = f'''<div style="margin-top:10px; background:#161b22; padding:10px; border-radius:8px;
             border-left:3px solid #58a6ff;">
  <div style="font-size:10px; color:#58a6ff; text-transform:uppercase;
              letter-spacing:0.5px; margin-bottom:4px;">PREDICTION FLIP</div>
  <div style="font-size:12px;">
    {init_evt['orig_label']} → <b style="color:#ff4757;">{done_evt['pred_label']}</b>
    <span style="color:#8b949e; margin-left:8px;">{len(loss_history)} steps · momentum μ=1.0</span>
  </div>
</div>'''

display(HTML(r3.render_variant_panel(
    variant_name='MI-FGSM (momentum)',
    paper_citation='Dong et al. 2018',
    description=(
        'PGD with gradient momentum (μ=1.0). Each step accumulates the running gradient: '
        'g_{t+1} = μ·g_t + ∇L/||∇L||₁. Perturbations transfer much better across model '
        'architectures/weights — best practice for attacking closed-source APIs.'
    ),
    image_columns=[
        {'label': 'CLEAN', 'image': orig_pil_view, 'caption': init_evt['orig_label']},
        {'label': 'δ (0.5-shifted)', 'image': delta_pil, 'caption': f'ε = 16/255'},
        {'label': 'ADVERSARIAL', 'image': adv_pil, 'caption': done_evt['pred_label']},
    ],
    extras_html=extras,
)))


### 4.4 DI²-FGSM (Xie et al. 2019)

MI-FGSM + Input Diversity. At each step, apply a random resize-and-pad to the input before computing the gradient. The optimizer learns perturbations that are robust to small spatial transforms, which **also** transfer better to other models. Today's best-practice transfer attack.


In [ ]:
from ch3 import variants
from ch3 import classifier_models as cm
from ch3.samples import load_sample
from ch3 import render as r3
from PIL import Image
from IPython.display import display, HTML

pil = load_sample(SAMPLE)
events = list(variants.di_fgsm_attack(clf, pil,
                                       epsilon=16/255, alpha=2/255,
                                       num_steps=20, mu=1.0, diversity_prob=0.5))
init_evt = events[0]; done_evt = events[-1]
loss_history = [e['loss'] for e in events if e['type'] == 'loss']

adv_pil_di = Image.fromarray((done_evt['x_adv_np'] * 255).clip(0, 255).astype('uint8'))
delta_pil_di = Image.fromarray((done_evt['delta_np'] * 255).clip(0, 255).astype('uint8'))
orig_pil_view = Image.fromarray((done_evt['x_orig_np'] * 255).clip(0, 255).astype('uint8'))

extras = f'''<div style="margin-top:10px; background:#161b22; padding:10px; border-radius:8px;
             border-left:3px solid #58a6ff;">
  <div style="font-size:10px; color:#58a6ff; text-transform:uppercase;
              letter-spacing:0.5px; margin-bottom:4px;">PREDICTION FLIP</div>
  <div style="font-size:12px;">
    {init_evt['orig_label']} → <b style="color:#ff4757;">{done_evt['pred_label']}</b>
    <span style="color:#8b949e; margin-left:8px;">{len(loss_history)} steps · diversity p=0.5</span>
  </div>
</div>'''

display(HTML(r3.render_variant_panel(
    variant_name='DI²-FGSM (momentum + input diversity)',
    paper_citation='Xie et al. 2019',
    description=(
        'MI-FGSM with random resize-and-pad applied to the input before each gradient. '
        'Perturbations robust to small spatial transforms also transfer better to other '
        'models. Today\'s best-practice transfer attack.'
    ),
    image_columns=[
        {'label': 'CLEAN', 'image': orig_pil_view, 'caption': init_evt['orig_label']},
        {'label': 'δ (0.5-shifted)', 'image': delta_pil_di, 'caption': f'ε = 16/255'},
        {'label': 'ADVERSARIAL', 'image': adv_pil_di, 'caption': done_evt['pred_label']},
    ],
    extras_html=extras,
)))


## Wrap-up

Tom's takeaways from Chapter 3:

1. **Open-source = white-box.** If your model weights are public, an attacker can compute the gradient of any loss with respect to the input. There is no preprocessing trick or text filter that fixes this.
2. **PGD is the canonical attack.** FGSM, C&W, DeepFool, SmoothFool all converge to similar perturbation budgets; PGD is the simplest, strongest, and most-used. Adapting PGD to a VLM just changes the loss — the optimizer is the same.
3. **No defense gets ASR below 10%.** Unlike Ch1 (where OCR gets you to 4%) and Ch2 (where LANCZOS gets you to 1%), adversarial perturbations exploit non-robust features in the vision encoder itself. **Stack 2-3 defenses** and accept clean-accuracy degradation.
4. **Transferability is the closed-source problem.** Adversarial images optimized on LLaVA transfer to GPT-4V at 15-25% ASR. MI-FGSM and DI-FGSM (slides 19, §4.3-4.4) push that higher. Don't assume a closed-source model is safe from white-box attacks on its open cousins.
5. **Universal perturbations and embedding attacks change the economics.** UAP: optimize once, apply forever. Embedding attack: no LLM gradient needed. Both significantly lower the attacker's per-target cost.
6. **Adversarial training is the only fundamental fix** — and it costs 10× training compute, drops clean accuracy by 5-8%, and is still imperfect. The research is active. Until then, defense-in-depth.

---

Next chapter: **Backdoored checkpoints** — what happens when Tom swaps to a faster open checkpoint that contains a hidden visual trigger. See `Ch4 - Backdoored Checkpoints.ipynb` (coming next).
